# Test avec CatBoostRegressor

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [2]:
df = pd.read_csv("../data/gold.csv")

In [3]:
# column_titles = df.columns.tolist()
# column_titles

In [4]:
X = df.drop(["prix_median"], axis=1)
y = df['prix_median']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=42)

categorial_features = ["proximité_autoroute"]

numerical_features = ['tx_crim', 'tx_residence','tx_commerce',
       'tx_nitriq', 'nb_piece', 'tx_ancienneté_parc_immo', 'distance_centre_emploi',
        'indice_impot_foncier',  'ratio_eleve_enseignant', 'tx_status_sociaux_eco_inf']

categorical_transformer = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorial_features),
        ('num', numerical_transformer, numerical_features)
    ],
    remainder="passthrough" 
)

# Créer un objet CatBoostRegressor
# model = CatBoostRegressor(random_state=42, verbose=False)
# Meilleurs hyperparamètres trouvés par Optuna (validation croisée, voir plus bas)
model = CatBoostRegressor(iterations = 900, learning_rate = 0.06314007319431346, depth = 4, loss_function = 'RMSE', subsample = 0.6, l2_leaf_reg = 9.263860413317243, random_strength = 3.017206838108549, random_state=42, verbose=False)


# Créer un pipeline avec le préprocesseur et le modèle CatBoostRegressor
pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', model)
])

# Entraîner le pipeline sur les données d'entraînement
pipe.fit(X_train, y_train)
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

print("Mean squared error : ")
print("TRAIN :",mean_squared_error(y_train, y_pred_train))
print("TEST :",mean_squared_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("Mean absolute error : ")
print("TRAIN :",mean_absolute_error(y_train, y_pred_train))
print("TEST :",mean_absolute_error(y_test, y_pred_test))

# print("/////////////////////////////////////////////////////")

print("R2 score : ")    
print("TRAIN :",r2_score(y_train, y_pred_train))
print("TEST :",r2_score(y_test, y_pred_test))

Mean squared error : 
TRAIN : 1.2544606627895531
TEST : 9.180970610645584
Mean absolute error : 
TRAIN : 0.8799320595629283
TEST : 1.9554165166445778
R2 score : 
TRAIN : 0.9855598997225703
TEST : 0.87480571416077


### Optuna (recherche des hyperparametres)

In [5]:
import os
import optuna

In [6]:
def objective(trial):
    # Définition des hyperparamètres à optimiser
    iterations = trial.suggest_int('iterations', 100, 1000, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    depth = trial.suggest_int('depth', 3, 10)
    loss_function = trial.suggest_categorical('loss_function', ['RMSE', 'MAE'])
    subsample = trial.suggest_float('subsample', 0.1, 1.0, step=0.1)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 0.1, 10.0)
    random_strength = trial.suggest_float('random_strength', 0.1, 10.0)
    
    # Création du modèle avec les hyperparamètres suggérés
    model = CatBoostRegressor(iterations=iterations,
                              learning_rate=learning_rate,
                              depth=depth,
                              loss_function=loss_function,
                              subsample=subsample,
                              l2_leaf_reg=l2_leaf_reg,
                              random_strength=random_strength,
                              random_state=42,
                              verbose=False,
                              allow_writing_files=False)
    
    # Même pipeline (préprocesseur + modèle) que pour l'évaluation finale
    pipe = Pipeline([
         ('preprocessor', preprocessor),
         ('model', model)
    ])
    
    # Validation croisée sur le jeu d'entraînement uniquement :
    # le jeu de test reste réservé à l'évaluation finale
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='r2', n_jobs=-1).mean()
    
    return r2

In [7]:
# Graine fixée pour que la recherche soit reproductible
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
# Nombre d'essais : 100 par défaut, réductible via la variable d'environnement N_TRIALS (CI)
n_trials = int(os.environ.get('N_TRIALS', 100))
study.optimize(objective, n_trials=n_trials)

[I 2026-10-08 13:32:54,721] A new study created in memory with name: no-name-16101b55-c467-4298-b309-cdf99aa23da6


[I 2026-10-08 13:32:58,296] Trial 0 finished with value: 0.860168134407373 and parameters: {'iterations': 400, 'learning_rate': 0.07969454818643935, 'depth': 8, 'loss_function': 'RMSE', 'subsample': 0.2, 'l2_leaf_reg': 0.6750277604651747, 'random_strength': 8.675143843171858}. Best is trial 0 with value: 0.860168134407373.


[I 2026-10-08 13:32:58,699] Trial 1 finished with value: 0.8734242376265857 and parameters: {'iterations': 700, 'learning_rate': 0.02607024758370768, 'depth': 3, 'loss_function': 'RMSE', 'subsample': 0.30000000000000004, 'l2_leaf_reg': 1.9000671753502962, 'random_strength': 1.915704647548995}. Best is trial 1 with value: 0.8734242376265857.


[I 2026-10-08 13:32:59,444] Trial 2 finished with value: 0.7888765319275208 and parameters: {'iterations': 400, 'learning_rate': 0.01120760621186057, 'depth': 6, 'loss_function': 'MAE', 'subsample': 0.2, 'l2_leaf_reg': 2.99223202049866, 'random_strength': 3.726982248607548}. Best is trial 1 with value: 0.8734242376265857.


[I 2026-10-08 13:32:59,897] Trial 3 finished with value: 0.837861542126338 and parameters: {'iterations': 500, 'learning_rate': 0.037183641805732096, 'depth': 4, 'loss_function': 'MAE', 'subsample': 0.1, 'l2_leaf_reg': 6.1146940338242395, 'random_strength': 1.7881888245041864}. Best is trial 1 with value: 0.8734242376265857.


...

[I 2026-10-08 13:34:14,502] Trial 95 finished with value: 0.8798127357076735 and parameters: {'iterations': 700, 'learning_rate': 0.043634704495912356, 'depth': 4, 'loss_function': 'RMSE', 'subsample': 0.7000000000000001, 'l2_leaf_reg': 9.57820903636331, 'random_strength': 3.784323198711548}. Best is trial 82 with value: 0.8832662704844811.


[I 2026-10-08 13:34:15,190] Trial 96 finished with value: 0.8787681172542804 and parameters: {'iterations': 800, 'learning_rate': 0.055909443606997046, 'depth': 4, 'loss_function': 'RMSE', 'subsample': 0.6, 'l2_leaf_reg': 9.49710910382149, 'random_strength': 4.149086071621777}. Best is trial 82 with value: 0.8832662704844811.


[I 2026-10-08 13:34:15,971] Trial 97 finished with value: 0.881749504151149 and parameters: {'iterations': 800, 'learning_rate': 0.08012797554850162, 'depth': 4, 'loss_function': 'RMSE', 'subsample': 0.7000000000000001, 'l2_leaf_reg': 9.077758076669845, 'random_strength': 3.3515176469661108}. Best is trial 82 with value: 0.8832662704844811.


[I 2026-10-08 13:34:16,628] Trial 98 finished with value: 0.8799999517835382 and parameters: {'iterations': 800, 'learning_rate': 0.08626240627996933, 'depth': 4, 'loss_function': 'RMSE', 'subsample': 0.6, 'l2_leaf_reg': 9.582785419141931, 'random_strength': 3.0063319389664414}. Best is trial 82 with value: 0.8832662704844811.


[I 2026-10-08 13:34:17,319] Trial 99 finished with value: 0.8781746557927722 and parameters: {'iterations': 700, 'learning_rate': 0.06542950177798411, 'depth': 4, 'loss_function': 'RMSE', 'subsample': 0.7000000000000001, 'l2_leaf_reg': 9.696284649521154, 'random_strength': 3.830335890993343}. Best is trial 82 with value: 0.8832662704844811.


In [8]:
best_params = study.best_params
best_value = study.best_value

In [9]:
print(best_params)
print(best_value)

{'iterations': 900, 'learning_rate': 0.06314007319431346, 'depth': 4, 'loss_function': 'RMSE', 'subsample': 0.6, 'l2_leaf_reg': 9.263860413317243, 'random_strength': 3.017206838108549}
0.8832662704844811


### Évaluation finale sur le jeu de test

In [10]:
# Pipeline avec les meilleurs hyperparamètres trouvés par Optuna,
# entraîné sur le jeu d'entraînement puis évalué une seule fois sur le jeu de test
best_pipe = Pipeline([
     ('preprocessor', preprocessor),
     ('model', CatBoostRegressor(**best_params, random_state=42, verbose=False))
])
best_pipe.fit(X_train, y_train)
y_pred_test = best_pipe.predict(X_test)

print("TEST MSE :", mean_squared_error(y_test, y_pred_test))
print("TEST MAE :", mean_absolute_error(y_test, y_pred_test))
print("TEST R2 :", r2_score(y_test, y_pred_test))

TEST MSE : 9.180970610645584
TEST MAE : 1.9554165166445778
TEST R2 : 0.87480571416077
